# Titanic Survival Prediction — the classic starter

This is the *"hello world"* of machine learning: given a passenger's details (age, sex, ticket class, ...), predict whether they **survived** the 1912 sinking.

The story behind the data is what makes it learnable. Lifeboats were scarce and loaded under a **"women and children first"** rule, and **1st-class** cabins were closer to the boat deck. So survival was *not* random — it leaned heavily toward being **female**, **young**, and in a **higher class**. A model can pick up exactly that signal.

Our plan, start to finish:
1. **Load** the data (with an offline fallback — see the note below).
2. **Explore** it — overall survival rate, then broken down by sex and class.
3. **Clean** it — fill in the missing ages and ports.
4. **Engineer** a few new features — family size, travelling-alone flag, an honorific *title*.
5. **Model** it — logistic regression in a pipeline, then read off what drove survival up or down.

> **A note on the dataset (please read).** The real Titanic dataset normally comes from **[Kaggle's Titanic competition](https://www.kaggle.com/c/titanic)** or from **seaborn** via `sns.load_dataset("titanic")` — but both need a network connection to download. This notebook is built to run **completely offline**, so if that download fails we fall back to a **synthetic, seeded stand-in**: a made-up DataFrame with the *same columns and the same real-world relationships* (female / young / higher-class ⇒ more likely to survive). Everything downstream works identically on either path, and the synthetic path is the one that runs here.

In [ ]:
# --- Standard scientific-Python toolkit ---
import numpy as np                 # numeric arrays + the random generator we seed below
import pandas as pd                # the DataFrame: our spreadsheet-in-code for the passenger table
import matplotlib.pyplot as plt    # low-level plotting (figures/axes)
import seaborn as sns              # high-level statistical charts (nice bar plots in one line)
import warnings                    # we silence a couple of cosmetic warnings for a clean run

warnings.filterwarnings("ignore")  # keep the notebook output tidy (no deprecation/pandas noise)

# --- Reproducibility: seed EVERYTHING so every run gives the exact same numbers ---
SEED = 42                          # one seed to rule them all
np.random.seed(SEED)               # seeds NumPy's legacy global RNG
rng = np.random.default_rng(SEED)  # a modern, self-contained NumPy Generator we'll use for synthetic data
sns.set_theme(style="whitegrid")   # a clean default look for the plots

print("Libraries loaded. Seed =", SEED)

## 1. Load the data (with an offline fallback)

We *try* the real dataset first. If the download fails (no network), the `except` block **synthesizes** a Titanic-like table instead. The trick that makes the synthetic data useful: we don't assign survival randomly — we build a realistic **probability** for each passenger from their sex, class, and age, then flip a biased coin. That way the model has genuine signal to find, just like the real thing.

In [ ]:
# We keep only the classic core columns that BOTH the real and synthetic tables share,
# so everything downstream behaves identically no matter which path ran.
CORE_COLS = ["survived", "pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]

try:
    # --- Preferred path: the real Titanic dataset (needs a network download) ---
    df = sns.load_dataset("titanic")          # raises if offline / not cached
    df = df[CORE_COLS].copy()                 # trim to the columns we'll use
    DATA_SOURCE = "REAL (seaborn/Kaggle Titanic)"

except Exception:
    # --- Offline fallback: synthesize a realistic, seeded Titanic-like table ---
    n = 891                                                   # same size as the real Titanic training set

    # 1) Draw the raw passenger attributes with plausible marginal distributions.
    pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])   # 3rd class was the biggest group
    sex    = rng.choice(["male", "female"], size=n, p=[0.65, 0.35]) # more men aboard than women
    age    = np.clip(rng.normal(29, 14, size=n), 0.4, 80).round(1)  # ~29yr mean, clipped to a sane range
    sibsp  = rng.poisson(0.5, size=n)                              # # siblings/spouses aboard (mostly 0-1)
    parch  = rng.poisson(0.4, size=n)                              # # parents/children aboard (mostly 0)

    # 2) Fare depends on class (1st class paid far more). Clip so nobody pays a negative fare.
    fare_mu = np.where(pclass == 1, 84.0, np.where(pclass == 2, 20.0, 13.0))  # mean fare per class
    fare    = np.clip(rng.normal(fare_mu, fare_mu * 0.25), 4.0, None).round(2)

    # 3) Port of embarkation: S (Southampton) most common, then C (Cherbourg), Q (Queenstown).
    embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

    # 4) THE IMPORTANT BIT — build a realistic survival probability, then sample survived from it.
    #    We work on the log-odds (logit) scale and add up effects, mirroring the real disaster.
    #    The effect sizes are deliberately strong so the classes separate well and the model
    #    has clear signal (less coin-flip noise near probability 0.5).
    logit  = -1.7                                     # baseline: most passengers did not survive
    logit += np.where(sex == "female", 3.2, 0.0)      # "women first" -> being female is a big boost
    logit += np.where(pclass == 1, 1.7, np.where(pclass == 2, 0.8, 0.0))  # higher class -> better odds
    logit += np.where(age < 15, 1.6, 0.0)             # "children first" -> a boost for the young
    logit += -0.020 * (age - 29)                      # a mild extra penalty for being older
    prob   = 1.0 / (1.0 + np.exp(-logit))             # squash log-odds into a 0-1 probability (sigmoid)
    survived = (rng.random(n) < prob).astype(int)     # flip a biased coin per passenger

    # 5) Assemble the DataFrame, then inject realistic MISSING values (so cleaning has real work to do).
    df = pd.DataFrame({
        "survived": survived, "pclass": pclass, "sex": sex, "age": age,
        "sibsp": sibsp, "parch": parch, "fare": fare, "embarked": embarked,
    })
    df.loc[rng.random(n) < 0.20, "age"] = np.nan       # ~20% of ages unknown (as in the real data)
    df.loc[rng.choice(n, size=2, replace=False), "embarked"] = np.nan  # a couple of missing ports
    DATA_SOURCE = "SYNTHETIC (offline stand-in)"

print("Data source:", DATA_SOURCE)
print("Shape (rows, cols):", df.shape)
df.head()

In [ ]:
# A quick health check before we touch anything.
# .info() shows each column's type and how many NON-null values it has -> reveals missing data.
df.info()

# Count missing values per column explicitly. We EXPECT gaps in 'age' and 'embarked'.
print("\nMissing values per column:")
print(df.isna().sum())

## 2. Exploratory Data Analysis (EDA)

Before modelling, always *look* at the data. Two questions tell most of the Titanic story:
- **Overall**, what fraction survived?
- Does survival differ by **sex** and by **class**? (Our hypothesis says: yes, strongly.)

In [ ]:
# 'survived' is coded 0/1, so its MEAN is exactly the survival RATE (fraction of 1s).
overall_rate = df["survived"].mean()
print(f"Overall survival rate: {overall_rate:.1%}")

# Group-wise survival rates. groupby(...)[col].mean() = average of 0/1 within each group = its survival rate.
print("\nSurvival rate by sex:")
print(df.groupby("sex")["survived"].mean().round(3))
print("\nSurvival rate by passenger class:")
print(df.groupby("pclass")["survived"].mean().round(3))

In [ ]:
# Two bar charts side by side. seaborn's barplot with a 0/1 target automatically
# plots the MEAN per group (i.e. the survival rate) with a small error bar.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: survival rate by sex. Expect the female bar to tower over the male bar.
sns.barplot(data=df, x="sex", y="survived", ax=axes[0], errorbar=None, palette="Set2")
axes[0].set_title("Survival rate by sex")
axes[0].set_ylabel("survival rate"); axes[0].set_ylim(0, 1)

# Right: survival rate by class. Expect a clear 1st > 2nd > 3rd staircase.
sns.barplot(data=df, x="pclass", y="survived", ax=axes[1], errorbar=None, palette="Set3")
axes[1].set_title("Survival rate by passenger class")
axes[1].set_ylabel("survival rate"); axes[1].set_ylim(0, 1)

plt.tight_layout(); plt.show()

**What we see.** The plots confirm the story: females survived at a far higher rate than males, and survival climbs steadily from 3rd → 2nd → 1st class. These two variables alone are strong predictors — a good sign that even a simple model will work. Now let's clean the data so we can feed it to one.

## 3. Clean the data

Models can't train on missing values, so we **impute** (fill in) the gaps:
- **Age** — fill each missing age with the **median age of its (sex, class) group**. Median (not mean) resists outliers, and going per-group is smarter than one global number: a 1st-class woman and a 3rd-class man have different typical ages.
- **Embarked** — only a couple missing, so fill with the **mode** (the single most common port). Simple and safe for a categorical column.

We keep every core column — each one is either a useful predictor or feeds a feature we build next.

In [ ]:
df_clean = df.copy()   # never mutate the original; work on a copy

# --- Impute AGE with the median age within each (sex, pclass) group ---
# transform("median") computes each group's median and broadcasts it back to every row in that group,
# so it lines up with df_clean row-for-row. We then only use it where age is missing.
group_median_age = df_clean.groupby(["sex", "pclass"])["age"].transform("median")
df_clean["age"] = df_clean["age"].fillna(group_median_age)
# Safety net: if some tiny group had NO known ages at all, fall back to the overall median.
df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())

# --- Impute EMBARKED with the mode (most frequent port) ---
embarked_mode = df_clean["embarked"].mode()[0]   # .mode() can return several; take the first
df_clean["embarked"] = df_clean["embarked"].fillna(embarked_mode)

# Confirm there are zero missing values left anywhere.
print("Missing values remaining:", int(df_clean.isna().sum().sum()))

## 4. Engineer a few features

Raw columns are fine, but a little domain knowledge lets us *create* more informative ones:
- **`family_size`** = `sibsp + parch + 1` (siblings/spouses + parents/children + the passenger themselves). One tidy number instead of two.
- **`is_alone`** = 1 when `family_size == 1`. Travelling alone had a different survival pattern than travelling with family.
- **`title`** — a social honorific (Mr / Mrs / Miss / Master). The real Titanic data hides this inside passenger *names*, which our columns don't contain, so we **derive** it heuristically from sex and age (e.g. a young male ⇒ *Master*, an adult male ⇒ *Mr*). Title neatly bundles sex + age + a hint of status.

We leave **age as a continuous number** (no binning) — the scaler in our pipeline handles its range, and logistic regression is happy with continuous inputs.

Finally we **one-hot encode** the categoricals (`sex`, `embarked`, `title`): turn each category into its own 0/1 column, because a linear model needs numbers, not text — and one-hot avoids implying a fake ordering.

In [ ]:
# --- family_size and is_alone ---
df_clean["family_size"] = df_clean["sibsp"] + df_clean["parch"] + 1   # +1 counts the passenger themselves
df_clean["is_alone"] = (df_clean["family_size"] == 1).astype(int)     # 1 if no family aboard, else 0

# --- title: derived from sex + age (since we have no name column to parse) ---
def make_title(row):
    if row["sex"] == "male":
        return "Master" if row["age"] < 15 else "Mr"     # boys -> Master, adult men -> Mr
    else:
        return "Miss" if row["age"] < 18 else "Mrs"      # girls -> Miss, adult women -> Mrs (a rough proxy)
df_clean["title"] = df_clean.apply(make_title, axis=1)   # axis=1 -> apply the function row by row

# --- Assemble the feature table X and the target y ---
# Drop sibsp/parch (now folded into family_size) to avoid redundancy; keep the rest.
feature_cols = ["pclass", "sex", "age", "fare", "embarked", "family_size", "is_alone", "title"]
X = df_clean[feature_cols].copy()
y = df_clean["survived"].copy()

# --- One-hot encode the text categoricals ---
# pd.get_dummies turns e.g. sex -> sex_male (0/1). drop_first drops one level per category to avoid
# the redundant, perfectly-correlated column (the 'dummy variable trap') that can upset a linear model.
X = pd.get_dummies(X, columns=["sex", "embarked", "title"], drop_first=True)

print("Engineered feature matrix shape:", X.shape)
print("Columns:", list(X.columns))
X.head()

## 5. Model — logistic regression in a pipeline

**Logistic regression** is the natural first model for a yes/no question: it learns a weight per feature, adds them up, and squashes the result through a sigmoid into a survival *probability*.

We wrap it in a **`Pipeline`** with a **`StandardScaler`**:
- The scaler puts every feature on a comparable scale (mean 0, variance 1). Without it, big-range `fare` would dwarf 0/1 flags, and regularization would be unfair.
- The pipeline learns the scaling **only from the training fold** and re-applies it automatically — no data leakage, no manual bookkeeping.

We split off **25% for testing**, `stratify`-ing on `survived` so the train and test sets keep the same survivor ratio.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Stratified 75/25 split: same survival ratio in both parts; seeded for reproducibility.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
print(f"train rows: {len(X_train)}   test rows: {len(X_test)}")

# Pipeline = scale, THEN fit logistic regression. Calling .fit on the pipeline does both in order.
model = Pipeline([
    ("scaler", StandardScaler()),                         # step 1: standardize features (fit on train only)
    ("logreg", LogisticRegression(max_iter=1000, random_state=SEED)),  # step 2: the classifier
])
model.fit(X_train, y_train)                               # learns scaling params + model weights together

# Predict on the held-out test set and measure plain accuracy (fraction of correct survive/not calls).
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"\nTest accuracy: {test_acc:.3f}")

### How good is it, really?

Accuracy alone can hide mistakes, so we also look at:
- **Confusion matrix** — a 2×2 grid of predicted vs. actual, showing *what kind* of errors we make (missed survivors vs. false alarms).
- **Classification report** — precision, recall, and F1 for each class ("did not survive" = 0, "survived" = 1).

In [ ]:
# --- Confusion matrix ---
# Rows = actual class, columns = predicted class. Diagonal = correct; off-diagonal = mistakes.
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

# Draw it as a labelled heatmap so the counts are easy to read.
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["pred: died", "pred: survived"],
            yticklabels=["actual: died", "actual: survived"])
plt.title("Confusion matrix (test set)")
plt.tight_layout(); plt.show()

# --- Per-class precision / recall / F1 ---
print("\nClassification report:\n")
print(classification_report(y_test, y_pred, target_names=["did not survive", "survived"]))

## 6. What did the model learn?

Logistic regression is wonderfully **interpretable**: each feature gets one weight (coefficient). Because we scaled the features first, the coefficients are directly comparable in size.
- A **positive** coefficient pushes the predicted survival probability **up**.
- A **negative** coefficient pushes it **down**.

We expect the story to reappear in the numbers: **female-linked titles** and **1st class** near the top (survival up), **being male**, **older age**, and **lower class** near the bottom.

In [ ]:
# Pull the fitted weights out of the pipeline's logistic-regression step.
coefs = model.named_steps["logreg"].coef_[0]     # one weight per feature (shape = n_features,)

# Pair each weight with its feature name and sort from most-positive to most-negative.
coef_table = (pd.DataFrame({"feature": X.columns, "coefficient": coefs})
                .sort_values("coefficient", ascending=False)
                .reset_index(drop=True))
print("Coefficients (sorted: survival-boosting at top, survival-hurting at bottom):\n")
print(coef_table.round(3).to_string(index=False))

# Visualize as a horizontal bar chart: green bars push survival up, red bars push it down.
plt.figure(figsize=(7, 4.5))
colors = ["#4c9f70" if c > 0 else "#c0524a" for c in coef_table["coefficient"]]
plt.barh(coef_table["feature"], coef_table["coefficient"], color=colors)
plt.axvline(0, color="k", linewidth=0.8)         # zero line: left = hurts, right = helps
plt.title("Logistic-regression coefficients (scaled features)")
plt.xlabel("effect on survival log-odds")
plt.gca().invert_yaxis()                           # keep the strongest positive at the top
plt.tight_layout(); plt.show()

## 7. Wrap-up

We took the Titanic table from raw rows to a working classifier:
1. **Loaded** the data with an offline synthetic fallback.
2. **Explored** it and saw survival rise for women and higher classes.
3. **Cleaned** it — group-median age, mode-filled port.
4. **Engineered** `family_size`, `is_alone`, and a derived `title`, then one-hot encoded the categoricals.
5. **Modelled** it with a scaler + logistic-regression pipeline, evaluated with accuracy, a confusion matrix, and a classification report, and **read off the coefficients** — which recovered the famous *women-and-children-first, higher-class-first* pattern.

**Where to go next:** cross-validation for a more stable accuracy estimate, a tree-based model (RandomForest / GradientBoosting) for non-linear interactions, and — with the real Kaggle data — parsing true titles out of passenger names.